### Utils file containing reussable functions

In [1]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install ffmpeg
!cd lerobot && pip install -e ".[train, dataset, smolvla]"

Cloning into 'lerobot'...
remote: Enumerating objects: 62550, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 62550 (delta 208), reused 173 (delta 173), pack-reused 62305 (from 2)
Receiving objects: 100% (62550/62550), 252.72 MiB | 38.70 MiB/s, done.
Resolving deltas: 100% (40079/40079), done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 1.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.

inspired from : https://colab.research.google.com/github/huggingface/notebooks/blob/main/lerobot/training-smolvla.ipynb#scrollTo=FMqvtVcThrdN

In [2]:
import os, getpass
os.environ["WANDB_API_KEY"] = getpass.getpass("Paste your W&B API key: ")

import wandb
wandb.login() 

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tjga-98 (tjga-98-epfl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import os, getpass
from huggingface_hub import login

token = getpass.getpass("Paste your HF token: ")
login(token=token)              # saves it for this machine, so later `!` commands find it
os.environ["HF_TOKEN"] = token

### Post-training SmolVLA using PEFT

In [4]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [8]:
# !lerobot-train \
#  --resume=true \
#  --config_path=TomasAnderegg/pill_sorting_post_trained

!lerobot-train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.optimizer_lr=1e-3 \
 --policy.device=cuda \
 --policy.scheduler_decay_lr=1e-4 \
 --policy.repo_id=TomasAnderegg/pill_sorting_post_trained_a100 \
 --num_workers=10  \
 --save_freq=3500 \
 --log_freq=100 \
 --policy.use_amp.mixed_precision=bf16  \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_merged_v2 \
 --steps=10000 \
 --batch_size=64 \
 --peft.method_type=LORA \
 --peft.r=64 \
 --peft.lora_alpha=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true 

config.json: 100% 2.30k/2.30k [00:00<00:00, 1.27MB/s]
INFO 2026-10-07 08:28:12 ot_train.py:490 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'reshard_after_forward': True,
                          'wrap_modules': None},
                 'grad_scaler': {'backoff_factor': 0.5,
                                 'growth_factor': 2.0,
                                 'growth_interval': 2000,
                           

In [6]:
 !nvidia-smi


: 

In [7]:
!nproc

12


Calculation of the number of parameters used with LoRa:
1. Check the target used for LoRa: "(model\\.vlm_with_expert\\.lm_expert\\..*\\.(q|v)_proj|model\\.(state_proj|action_in_proj|action_out_proj|action_time_mlp_in|action_time_mlp_out))"
2. For each of them extract the features_in and features_out
3. Since LoRa adds 2 matrices A with size r x features_in and B with size features_out x r, we can calculate the total number for each adapted target.
4. Add everything

In [1]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
print(policy)

/home/tjga98/miniconda3/envs/lerobot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tjga98/miniconda3/envs/lerobot/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


`torch_dtype` is deprecated! Use `dtype` instead!


KeyboardInterrupt: 

In [5]:
#Full post-training:
!lerobot-train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=false \
 --policy.freeze_vision_encoder=false \
 --policy.optimizer_lr=1e-4 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=20000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_full_ft_a100 \
 --job_name=pill_smolvla_full_ft \
 --num_workers=10 \
 --save_freq=5000 \
 --log_freq=100 \
 --policy.use_amp.mixed_precision=bf16  \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_merged_v2 \
 --steps=20000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true

config.json: 100% 2.30k/2.30k [00:00<00:00, 3.62MB/s]
INFO 2026-10-15 22:36:01 ot_train.py:581 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'reshard_after_forward': True,
                          'wrap_modules': None},
                 'grad_scaler': {'backoff_factor': 0.5,
                                 'growth_factor': 2.0,
                                 'growth_interval': 2000,
                           

In [ ]:
## Fine tuning only the action expert
!lerobot-train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=true \
 --policy.freeze_vision_encoder=true \
 --policy.optimizer_lr=1e-4 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=20000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_expert_only_t4_better_data \
 --job_name=pill_smolvla_expert_only \
 --num_workers=10 \
 --save_freq=5000 \
 --log_freq=100 \
 --policy.use_amp.mixed_precision=bf16 \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_v3 \
 --steps=20000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true

config.json: 100% 2.30k/2.30k [00:00<00:00, 1.74MB/s]
INFO 2026-10-22 21:29:50 ot_train.py:582 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'reshard_after_forward': True,
                          'wrap_modules': None},
                 'grad_scaler': {'backoff_factor': 0.5,
                                 'growth_factor': 2.0,
                                 'growth_interval': 2000,
                           

In [ ]:
## Fine tuning only the action expert using mixed precision: use accelerate 
!accelerate launch \
  --mixed_precision=bf16 \
    -m lerobot.scripts.lerobot_train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=true \
 --policy.freeze_vision_encoder=true \
 --policy.optimizer_lr=1e-4 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=20000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_expert_only_A100_big_data \
 --job_name=pill_smolvla_expert_only_v3 \
 --num_workers=10 \
 --save_freq=5000 \
 --log_freq=100 \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_v3 \
 --steps=20000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true